In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG, SCHEMA = "payment_lakehouse", "pipeline"
LANDING    = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
CHECKPOINT = f"{LANDING}/_checkpoints"

spark.sql(f"USE CATALOG {CATALOG}"); spark.sql(f"USE SCHEMA {SCHEMA}")

VALID_REGIONS    = ["IN","US","EU","SG","AE"]
VALID_CURRENCIES = ["INR","USD","EUR","SGD","AED"]

In [0]:
# Targets created up front so MERGE has something to merge into.
for t, extra in [("silver_payments", ""), ("silver_invoices", "")]:
    spark.sql(f"""
      CREATE TABLE IF NOT EXISTS {t} (
        event_id STRING, event_type STRING, event_ts TIMESTAMP, event_dt DATE,
        payment_id STRING, customer_id STRING, invoice_id STRING,
        region STRING, currency STRING, amount DOUBLE, payment_method STRING,
        ingest_ts TIMESTAMP, source_file STRING
      ) USING DELTA PARTITIONED BY (event_dt)""")

spark.sql("""
  CREATE TABLE IF NOT EXISTS silver_quarantine (
    event_id STRING, event_type STRING, event_ts TIMESTAMP,
    payment_id STRING, region STRING, currency STRING, amount DOUBLE,
    reject_reason STRING, ingest_ts TIMESTAMP, source_file STRING
  ) USING DELTA""")
print("targets ready")

In [0]:
dbutils.fs.rm(f"{CHECKPOINT}/silver", True)   # stream failed; clear state

In [0]:
# try_cast returns NULL instead of throwing, which lets a bad value be
# quarantined rather than killing the stream.
stream = (spark.readStream.table("bronze_events")
    .withColumn("event_ts",   F.to_timestamp("event_ts"))
    .withColumn("event_dt",   F.to_date("event_ts"))
    .withColumn("amount_num", F.expr("try_cast(amount AS DOUBLE)"))
    .withWatermark("event_ts", "7 days")
    .dropDuplicatesWithinWatermark(["event_id"]))

reject_reason = (
    F.when(F.col("event_id").isNull(), F.lit("missing_field:event_id"))
     .when(~F.col("event_type").isin("PAYMENT_RECEIVED","INVOICE_CLEARED"),
           F.lit("unknown_event_type"))
     .when(F.col("amount").isNull(), F.lit("null_field:amount"))
     .when(F.col("amount_num").isNull(), F.lit("amount_not_numeric"))
     .when(F.col("amount_num") <= 0, F.lit("amount_not_positive"))
     .when(F.col("currency").isNull() | ~F.col("currency").isin(VALID_CURRENCIES),
           F.concat(F.lit("unknown_currency:"), F.coalesce(F.col("currency"), F.lit("null"))))
     .when(F.col("region").isNull(), F.lit("missing_field:region"))
     .when(~F.col("region").isin(VALID_REGIONS),
           F.concat(F.lit("unknown_region:"), F.col("region")))
     .when(F.col("payment_id").isNull(), F.lit("missing_field:payment_id"))
     .when((F.col("event_type")=="INVOICE_CLEARED") & F.col("invoice_id").isNull(),
           F.lit("missing_field:invoice_id"))
     .otherwise(F.lit(None)))

validated = stream.withColumn("reject_reason", reject_reason)

In [0]:
SILVER_SEL = ["event_id","event_type","event_ts","event_dt","payment_id",
              "customer_id","invoice_id","region","currency",
              F.col("amount_num").alias("amount"),
              "payment_method","ingest_ts","source_file"]
QUAR_SEL   = ["event_id","event_type","event_ts","payment_id","region","currency",
              F.col("amount_num").alias("amount"),
              "reject_reason","ingest_ts","source_file"]

def upsert(batch_df, batch_id):
    good = batch_df.filter(F.col("reject_reason").isNull())
    bad  = batch_df.filter(F.col("reject_reason").isNotNull())

    def merge(df, table, sel):
        if df.isEmpty(): return
        (DeltaTable.forName(spark, table).alias("t")
            .merge(df.select(*sel).alias("s"), "t.event_id = s.event_id")
            .whenNotMatchedInsertAll()
            .execute())

    merge(good.filter(F.col("event_type")=="PAYMENT_RECEIVED"), "silver_payments", SILVER_SEL)
    merge(good.filter(F.col("event_type")=="INVOICE_CLEARED"),  "silver_invoices", SILVER_SEL)
    merge(bad, "silver_quarantine", QUAR_SEL)
    print(f"batch {batch_id} done")

In [0]:
# foreachBatch: the native Delta sink only appends. This hands each
# micro-batch over as a normal DataFrame so it can MERGE instead.
(validated.writeStream
    .foreachBatch(upsert)
    .option("checkpointLocation", f"{CHECKPOINT}/silver")
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

print("silver loaded")

In [0]:
display(spark.sql("""
  SELECT 'payments' t, count(*) n FROM silver_payments
  UNION ALL SELECT 'invoices', count(*) FROM silver_invoices
  UNION ALL SELECT 'quarantine', count(*) FROM silver_quarantine
"""))
display(spark.sql("SELECT reject_reason, count(*) n FROM silver_quarantine GROUP BY 1 ORDER BY 2 DESC"))